# Vector intersection lab

Extract one page's raw vectors and cluster them exactly as P3 **VectorClassification** does. The chain is
layer -> color -> stroke-width buckets -> seqno-overlap merge -> constrained spatial clustering, via
`classify_vectors`.

Then, **within each cluster**, count for every `Vector` (one whole `get_drawings()` drawing) how many
**segments** of *other* Vectors in that cluster **properly cross** it.
- A segment is a flattened piece: an `'l'`, one `'re'`/`'qu'` edge, or one of the `CURVE_STEPS` pieces of a
  `'c'`.
- A foreign segment that crosses two of this Vector's segments counts once.
- Its own segments never count.
- Only interior X-crossings count; touching endpoints, corners and T-junctions don't.

Colouring is per Vector:
- `gradient / crossing segments`: linear min..max over every scored vector, on the ramp
  red -> yellow -> green -> blue.
- `buckets / 0 crossings` (grey), plus **5 percentile buckets** (quintiles) computed over the vectors with
  >= 1 crossing. Each bucket is coloured by sampling the same ramp at evenly spaced points. Tied counts
  never split across buckets, so heavy ties can leave fewer than 5.

Writes a `scripts/pipeline_report_viewer.py`-compatible folder under `outputs/vector_intersection_lab/`, plus a
histogram. Helpers live in `rastervec/notebooks/_vector_probe_helpers.py`.

## 0 - Config

In [ ]:
PDF_PATH = None     # None -> first PDF under references/
PAGE = 0
CROSS_EPS = 0.01    # pt; an endpoint closer than this to the other segment's line is "touching", not crossing
CURVE_STEPS = 8     # segments per flattened 'c' item (each piece counts as one crossing segment)
N_BUCKETS = 5       # percentile buckets over vectors with >= 1 crossing
LINE_WIDTH = 1.0    # min overlay stroke width (pt) in the debug PDFs

## 1 - Bootstrap

In [ ]:
import colorsys
import os, sys
from pathlib import Path

_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import matplotlib.pyplot as plt
import numpy as np

from rastervec.commons.helpers.geometry import union_bbox
from rastervec.notebooks import _vector_probe_helpers as h

if PDF_PATH is None:
    refs = sorted((Path(_root) / 'references').glob('*.pdf'))
    assert refs, 'no PDFs found under references/'
    PDF_PATH = str(refs[0])
print(PDF_PATH, 'page', PAGE)

## 2 - Extract + cluster (P3 VectorClassification chain)

In [ ]:
page_meta, vectors = h.load_page_vectors(PDF_PATH, PAGE)
clusters = h.cluster_vectors(vectors, page_meta)
print(f'{len(vectors)} vectors -> {len(clusters)} clusters '
      f'across {len({k for k, _ in clusters})} layer/color/width buckets')

## 3 - Crossing segments per Vector, within each cluster

In [ ]:
seg_crossings = {}   # id(v) -> distinct foreign segments crossing v
records = []         # (cluster_index, bucket_key, vector, count)
for ci, (key, members) in enumerate(clusters):
    for v, n in zip(members, h.crossing_segment_counts(members, eps=CROSS_EPS, curve_steps=CURVE_STEPS)):
        seg_crossings[id(v)] = n
        records.append((ci, key, v, n))

all_counts = [n for *_, n in records]
lo, hi = (min(all_counts), max(all_counts)) if all_counts else (0, 0)
n_zero = sum(1 for n in all_counts if n == 0)
print(f'{len(records)} vectors scored; crossing segments range {lo}..{hi}; {n_zero} with 0 crossings')

## 4 - Debug layers

In [ ]:
report = h.DebugReport('vector_intersection_lab', PDF_PATH, page_meta, line_width=LINE_WIDTH)
all_vectors = [v for _, _, v, _ in records]

# Gradient: linear min..max, drawn low-to-high so the busiest vectors end up on top.
grad_order = sorted(all_vectors, key=lambda v: seg_crossings[id(v)])
report.add_vectors('gradient', f'crossing segments ({lo}-{hi}, min-max)', h.ramp4(1.0), grad_order,
                   lambda v: h.ramp4(h.normalise(seg_crossings[id(v)], lo, hi)))

# Buckets: zero layer + percentile buckets over the vectors with >= 1 crossing.
report.add_vectors('buckets', '0 crossing segments', h.GREY,
                   [v for v in all_vectors if seg_crossings[id(v)] == 0], lambda v: h.GREY)
nonzero = [v for v in all_vectors if seg_crossings[id(v)] > 0]
buckets = h.percentile_buckets([seg_crossings[id(v)] for v in nonzero], N_BUCKETS)
if len(buckets) < N_BUCKETS:
    print(f'note: only {len(buckets)} of {N_BUCKETS} percentile buckets survive (tied counts share a bucket)')
bucket_edges = []
for qi, (b_lo, b_hi, idxs) in enumerate(buckets):
    rgb = h.ramp4(qi / max(1, len(buckets) - 1))
    span = f'{b_lo:g}' if b_lo == b_hi else f'{b_lo:g}-{b_hi:g}'
    noun = 'crossing segment' if b_lo == b_hi == 1 else 'crossing segments'
    report.add_vectors('buckets', f'Q{qi + 1}: {span} {noun} ({len(idxs)} vectors)', rgb,
                       [nonzero[i] for i in idxs], lambda v, rgb=rgb: rgb)
    bucket_edges.append(b_hi)
    print(f'  Q{qi + 1}: {span:>8} {noun}, {len(idxs)} vectors')

hues = h.golden_hues(len(clusters))
report.add_boxes('clusters', 'cluster bboxes', '#888888', [
    (union_bbox([v.bbox for v in members]), colorsys.hsv_to_rgb(hues[ci], 0.8, 0.8))
    for ci, (_, members) in enumerate(clusters)
])

## 5 - Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
if hi <= 60:
    xs = np.arange(lo, hi + 1)
    heights = [all_counts.count(x) for x in xs]
    ax.bar(xs, heights, color=[h.ramp4(h.normalise(x, lo, hi)) for x in xs],
           edgecolor='black', linewidth=0.4)
else:
    _, bins, patches = ax.hist(all_counts, bins=40, edgecolor='black', linewidth=0.4)
    for left, right, patch in zip(bins[:-1], bins[1:], patches):
        patch.set_facecolor(h.ramp4(h.normalise((left + right) / 2, lo, hi)))
for edge in bucket_edges[:-1]:
    ax.axvline(edge + 0.5, color='black', linestyle='--', linewidth=0.8)
ax.set_yscale('symlog')
ax.set_xlabel('distinct crossing segments of other Vectors (within cluster)')
ax.set_ylabel('vectors (symlog)')
ax.set_title(f'Crossing segments per Vector -- {Path(PDF_PATH).stem} p{PAGE} '
             f'({n_zero} at 0; dashed = quintile edges over >0)')
fig.tight_layout()
report.save_figure(fig, 'histogram_crossing_segments')
plt.show()

## 6 - Per-cluster summary (top 15 by mean crossing segments, clusters with >= 3 vectors)

In [ ]:
by_cluster = {}
for ci, key, v, n in records:
    by_cluster.setdefault(ci, (key, []))[1].append(n)
rows = sorted(((np.mean(ns), ci, key, ns) for ci, (key, ns) in by_cluster.items() if len(ns) >= 3),
              key=lambda r: -r[0])[:15]
print(f"{'cluster':>7} {'n':>5} {'mean':>7} {'max':>5}  bucket (layer, color, width)")
for mean, ci, key, ns in rows:
    print(f'{ci:>7} {len(ns):>5} {mean:>7.2f} {max(ns):>5}  {key}')

## 7 - Write manifest

In [ ]:
out = report.finish()
print('wrote', out)
for e in report.layers:
    print(f"  {e['stage']:>14} | {e['layer']}")
print()
print('open with:')
print(report.viewer_command())